In [2]:
import torch
import kagglehub
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torchvision.models import resnet18
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import cv2
from tqdm import tqdm
import math
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

path = kagglehub.dataset_download("kushagrapandya/visdrone-dataset")
path = Path(path)

train_images_path = path / "VisDrone2019-DET-train" / "VisDrone2019-DET-train" / "images"
train_ann_path = path / "VisDrone2019-DET-train" / "VisDrone2019-DET-train" / "annotations"
val_images_path = path / "VisDrone2019-DET-val" / "VisDrone2019-DET-val" / "images"
val_ann_path = path / "VisDrone2019-DET-val" / "VisDrone2019-DET-val" / "annotations"


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


In [3]:
class VisDroneDataset(Dataset):
    def __init__(self, img_dir, ann_dir, transform=None, max_samples=None):
        self.img_dir = Path(img_dir)
        self.ann_dir = Path(ann_dir)
        self.transform = transform
        self.samples = []
        self.class_map = {3: 0, 4: 1, 5: 2, 8: 3}
        
        if not self.img_dir.exists():
            print(f"Папка не найдена: {self.img_dir}")
            return
        
        images = list(self.img_dir.glob("*.jpg")) + list(self.img_dir.glob("*.JPG"))
        print(f"Найдено {len(images)} изображений")
        
        count = 0
        for img_path in images:
            ann_path = self.ann_dir / f"{img_path.stem}.txt"
            if not ann_path.exists():
                continue
            
            img = cv2.imread(str(img_path))
            if img is None:
                continue
            h, w = img.shape[:2]
            
            boxes = []
            labels = []
            
            try:
                with open(ann_path, 'r') as f:
                    for line in f:
                        parts = line.strip().split(',')
                        if len(parts) < 8:
                            continue
                        
                        x, y, bw, bh, score, cls_id = [int(p) for p in parts[:6]]
                        
                        if score == 0 or cls_id not in self.class_map:
                            continue
                        
                        cx = (x + bw/2) / w
                        cy = (y + bh/2) / h
                        nw = bw / w
                        nh = bh / h
                        
                        boxes.append([cx, cy, nw, nh])
                        labels.append(self.class_map[cls_id])
                        
            except Exception as e:
                continue
                
            if len(boxes) > 0:
                self.samples.append({
                    'img_path': str(img_path),
                    'boxes': torch.tensor(boxes, dtype=torch.float32),
                    'labels': torch.tensor(labels, dtype=torch.int64)
                })
                count += 1
                
            if max_samples and count >= max_samples:
                break
                
        print(f"Загружено {len(self.samples)} изображений")
        
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        img = cv2.imread(sample['img_path'])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        if self.transform:
            img = self.transform(img)
            
        return img, {'boxes': sample['boxes'], 'labels': sample['labels']}

In [4]:
train_transform = T.Compose([
    T.ToPILImage(),
    T.Resize((640, 640)),
    T.RandomHorizontalFlip(0.5),
    T.ColorJitter(0.2, 0.2, 0.2),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = T.Compose([
    T.ToPILImage(),
    T.Resize((640, 640)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

def collate_fn(batch):
    images = torch.stack([b[0] for b in batch])
    targets = [b[1] for b in batch]
    return images, targets

train_dataset = VisDroneDataset(train_images_path, train_ann_path, transform=train_transform)
val_dataset = VisDroneDataset(val_images_path, val_ann_path, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn, num_workers=2)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

Найдено 6471 изображений
Загружено 6271 изображений
Найдено 548 изображений
Загружено 535 изображений
Train batches: 196, Val batches: 17


In [5]:
class SSD(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        self.num_classes = num_classes
        
        backbone = resnet18(weights='DEFAULT')
        self.backbone = nn.Sequential(*list(backbone.children())[:-2])
        
        self.conv1 = nn.Conv2d(512, 256, 1)
        self.conv2 = nn.Conv2d(256, 512, 3, stride=2, padding=1)
        self.conv3 = nn.Conv2d(512, 256, 1)
        self.conv4 = nn.Conv2d(256, 512, 3, stride=2, padding=1)
        self.conv5 = nn.Conv2d(512, 256, 1)
        self.conv6 = nn.Conv2d(256, 512, 3, stride=2, padding=1)
        
        self.loc1 = nn.Conv2d(512, 4 * 4, 3, padding=1)
        self.loc2 = nn.Conv2d(512, 6 * 4, 3, padding=1)
        self.loc3 = nn.Conv2d(512, 6 * 4, 3, padding=1)
        self.loc4 = nn.Conv2d(512, 4 * 4, 3, padding=1)
        
        self.conf1 = nn.Conv2d(512, 4 * num_classes, 3, padding=1)
        self.conf2 = nn.Conv2d(512, 6 * num_classes, 3, padding=1)
        self.conf3 = nn.Conv2d(512, 6 * num_classes, 3, padding=1)
        self.conf4 = nn.Conv2d(512, 4 * num_classes, 3, padding=1)
        
    def forward(self, x):
        outs = []
        
        x = self.backbone(x)
        outs.append(x)
        
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        outs.append(x)
        
        x = torch.relu(self.conv3(x))
        x = torch.relu(self.conv4(x))
        outs.append(x)
        
        x = torch.relu(self.conv5(x))
        x = torch.relu(self.conv6(x))
        outs.append(x)
        
        locs, confs = [], []
        
        loc = self.loc1(outs[0]).permute(0, 2, 3, 1).contiguous()
        loc = loc.view(loc.size(0), -1, 4)
        conf = self.conf1(outs[0]).permute(0, 2, 3, 1).contiguous()
        conf = conf.view(conf.size(0), -1, self.num_classes)
        locs.append(loc)
        confs.append(conf)
        
        loc = self.loc2(outs[1]).permute(0, 2, 3, 1).contiguous()
        loc = loc.view(loc.size(0), -1, 4)
        conf = self.conf2(outs[1]).permute(0, 2, 3, 1).contiguous()
        conf = conf.view(conf.size(0), -1, self.num_classes)
        locs.append(loc)
        confs.append(conf)
        
        loc = self.loc3(outs[2]).permute(0, 2, 3, 1).contiguous()
        loc = loc.view(loc.size(0), -1, 4)
        conf = self.conf3(outs[2]).permute(0, 2, 3, 1).contiguous()
        conf = conf.view(conf.size(0), -1, self.num_classes)
        locs.append(loc)
        confs.append(conf)
        
        loc = self.loc4(outs[3]).permute(0, 2, 3, 1).contiguous()
        loc = loc.view(loc.size(0), -1, 4)
        conf = self.conf4(outs[3]).permute(0, 2, 3, 1).contiguous()
        conf = conf.view(conf.size(0), -1, self.num_classes)
        locs.append(loc)
        confs.append(conf)
        
        return torch.cat(locs, 1), torch.cat(confs, 1)

In [6]:
def calc_iou(boxes1, boxes2):
    def to_xyxy(b):
        cx, cy, w, h = b[..., 0], b[..., 1], b[..., 2], b[..., 3]
        return torch.stack([cx - w/2, cy - h/2, cx + w/2, cy + h/2], dim=-1)
    
    b1 = to_xyxy(boxes1)
    b2 = to_xyxy(boxes2)
    
    b1 = b1.unsqueeze(1)
    b2 = b2.unsqueeze(0)
    
    inter_x1 = torch.max(b1[..., 0], b2[..., 0])
    inter_y1 = torch.max(b1[..., 1], b2[..., 1])
    inter_x2 = torch.min(b1[..., 2], b2[..., 2])
    inter_y2 = torch.min(b1[..., 3], b2[..., 3])
    
    inter_w = torch.clamp(inter_x2 - inter_x1, min=0)
    inter_h = torch.clamp(inter_y2 - inter_y1, min=0)
    inter = inter_w * inter_h
    
    area1 = (b1[..., 2] - b1[..., 0]) * (b1[..., 3] - b1[..., 1])
    area2 = (b2[..., 2] - b2[..., 0]) * (b2[..., 3] - b2[..., 1])
    union = area1 + area2 - inter
    
    return inter / torch.clamp(union, min=1e-6)

def generate_ssd_anchors():
    anchors = []
    scales = [(40, 40, 4), (20, 20, 6), (10, 10, 6), (5, 5, 4)]
    sizes = [24, 48, 96, 128]
    
    for (h, w, num_anchors), size in zip(scales, sizes):
        for y in range(h):
            for x in range(w):
                cx = (x + 0.5) / w
                cy = (y + 0.5) / h
                
                if num_anchors == 4:
                    ratios = [1, 2, 0.5, 1]
                else:
                    ratios = [1, 2, 0.5, 1/3, 3, 1]
                
                for ratio in ratios[:num_anchors]:
                    w_anchor = size * math.sqrt(ratio) / 640
                    h_anchor = size / math.sqrt(ratio) / 640
                    anchors.append([cx, cy, w_anchor, h_anchor])
    
    return torch.tensor(anchors, dtype=torch.float32)

ssd_anchors = generate_ssd_anchors()
print(f"Anchors: {len(ssd_anchors)}")

Anchors: 9500


In [7]:
def ssd_multibox_loss(loc_preds, conf_preds, anchors, targets, pos_thresh=0.5, neg_thresh=0.3, pos_neg_ratio=3):
    batch_size = loc_preds.size(0)
    total_loc = 0.0
    total_conf = 0.0
    n_pos_total = 0
    
    smooth_l1 = nn.SmoothL1Loss(reduction='sum')
    cross_entropy = nn.CrossEntropyLoss(reduction='sum')
    
    for i in range(batch_size):
        gt_boxes = targets[i]['boxes'].to(loc_preds.device)
        gt_labels = targets[i]['labels'].to(loc_preds.device)
        
        if gt_boxes.size(0) == 0:
            continue
        
        num_anchors = loc_preds.shape[1]
        anchors_batch = anchors[:num_anchors].to(loc_preds.device)
        
        ious = calc_iou(anchors_batch, gt_boxes)
        best_iou, best_gt_idx = ious.max(dim=1)
        
        best_iou_per_gt, best_anchor_per_gt = ious.max(dim=0)
        best_iou[best_anchor_per_gt] = 1.0
        
        pos_mask = best_iou >= pos_thresh
        n_pos = pos_mask.sum().item()
        
        if n_pos == 0:
            continue
            
        n_pos_total += n_pos
        
        matched_gt = gt_boxes[best_gt_idx[pos_mask]]
        anchors_pos = anchors_batch[pos_mask]
        
        tgt_loc = torch.zeros_like(loc_preds[i][pos_mask])
        tgt_loc[:, 0] = (matched_gt[:, 0] - anchors_pos[:, 0]) / (anchors_pos[:, 2] + 1e-6)
        tgt_loc[:, 1] = (matched_gt[:, 1] - anchors_pos[:, 1]) / (anchors_pos[:, 3] + 1e-6)
        tgt_loc[:, 2] = torch.log((matched_gt[:, 2] + 1e-6) / (anchors_pos[:, 2] + 1e-6))
        tgt_loc[:, 3] = torch.log((matched_gt[:, 3] + 1e-6) / (anchors_pos[:, 3] + 1e-6))
        
        total_loc += smooth_l1(loc_preds[i][pos_mask], tgt_loc)
        
        neg_mask = (best_iou < neg_thresh) & (~pos_mask)
        
        if neg_mask.sum() > 0:
            conf_loss_all = torch.zeros(conf_preds.shape[1], device=conf_preds.device)
            for c in range(conf_preds.shape[1]):
                if conf_preds[i][c].max() > -100:
                    conf_loss_all[c] = cross_entropy(conf_preds[i][c:c+1], torch.tensor([0], device=conf_preds.device))
            
            neg_losses = conf_loss_all[neg_mask]
            _, neg_order = neg_losses.sort(descending=True)
            n_neg = min(neg_mask.sum().item(), int(n_pos * pos_neg_ratio))
            neg_selected = torch.zeros_like(neg_mask)
            neg_indices = torch.where(neg_mask)[0][neg_order[:n_neg]]
            neg_selected[neg_indices] = True
        else:
            neg_selected = neg_mask
        
        conf_mask = pos_mask | neg_selected
        conf_targets = torch.zeros(conf_preds.shape[1], dtype=torch.long, device=conf_preds.device)
        conf_targets[pos_mask] = gt_labels[best_gt_idx[pos_mask]] + 1
        
        total_conf += cross_entropy(conf_preds[i][conf_mask], conf_targets[conf_mask])
    
    if n_pos_total > 0:
        return (total_loc / n_pos_total) + (total_conf / n_pos_total)
    else:
        return torch.tensor(0.0, device=loc_preds.device, requires_grad=True)

In [8]:
accumulation_steps = 4

def train_ssd(model, train_loader, val_loader, anchors, epochs=20, accumulation_steps=4):
    optimizer = optim.AdamW(model.parameters(), lr=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, epochs)
    history = {'train_loss': [], 'val_loss': []}
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        optimizer.zero_grad()
        
        for i, (imgs, targets) in enumerate(tqdm(train_loader, desc=f"SSD Epoch {epoch+1}/{epochs}")):
            imgs = imgs.to(device)
            for t in targets:
                t['boxes'] = t['boxes'].to(device)
                t['labels'] = t['labels'].to(device)
            
            locs, confs = model(imgs)
            loss = ssd_multibox_loss(locs, confs, anchors, targets)
            loss = loss / accumulation_steps
            loss.backward()
            
            if (i + 1) % accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 0.1)
                optimizer.step()
                optimizer.zero_grad()
            
            train_loss += loss.item() * accumulation_steps
        
        if (i + 1) % accumulation_steps != 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.1)
            optimizer.step()
            optimizer.zero_grad()
        

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for imgs, targets in val_loader:
                imgs = imgs.to(device)
                for t in targets:
                    t['boxes'] = t['boxes'].to(device)
                    t['labels'] = t['labels'].to(device)
                locs, confs = model(imgs)
                loss = ssd_multibox_loss(locs, confs, anchors, targets)
                val_loss += loss.item()
        
        scheduler.step()
        avg_train = train_loss / len(train_loader)
        avg_val = val_loss / len(val_loader)
        history['train_loss'].append(avg_train)
        history['val_loss'].append(avg_val)
        print(f"SSD Epoch {epoch+1} | Train Loss: {avg_train:.4f} | Val Loss: {avg_val:.4f}")
    
    return history

In [ ]:
ssd_model = SSD(num_classes=5).to(device)
print(f"SSD parameters: {sum(p.numel() for p in ssd_model.parameters()):,}")

ssd_history = train_ssd(ssd_model, train_loader, val_loader, ssd_anchors, epochs=20, accumulation_steps=4)

torch.save(ssd_model.state_dict(), 'ssd_model.pth')

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 183MB/s]


SSD parameters: 15,940,596


SSD Epoch 1/20:  64%|██████▍   | 125/196 [33:37<19:01, 16.07s/it]

In [ ]:
class DETR(nn.Module):
    def __init__(self, num_classes=5, hidden_dim=256, num_queries=50):
        super().__init__()
        self.num_queries = num_queries
        self.num_classes = num_classes
        
        backbone = resnet18(weights='DEFAULT')
        self.backbone = nn.Sequential(*list(backbone.children())[:-2])
        
        self.proj = nn.Conv2d(512, hidden_dim, 1)
        self.pos_embed = nn.Parameter(torch.randn(1, 1600, hidden_dim))
        
        self.transformer = nn.Transformer(
            d_model=hidden_dim,
            nhead=8,
            num_encoder_layers=3,
            num_decoder_layers=3,
            dim_feedforward=2048,
            dropout=0.1,
            batch_first=True
        )
        
        self.query_embed = nn.Embedding(num_queries, hidden_dim)
        self.class_head = nn.Linear(hidden_dim, num_classes)
        self.bbox_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 4),
            nn.Sigmoid()
        )
        
    def forward(self, x):
        features = self.backbone(x)
        b, c, h, w = features.shape
        
        features = self.proj(features)
        features = features.flatten(2).permute(0, 2, 1)
        
        pos = self.pos_embed[:, :h*w, :].expand(b, -1, -1)
        queries = self.query_embed.weight.unsqueeze(0).expand(b, -1, -1)
        
        hidden = self.transformer(features + pos, queries)
        
        class_logits = self.class_head(hidden)
        bbox_coords = self.bbox_head(hidden)
        
        return bbox_coords, class_logits

class DETRLoss(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.num_classes = num_classes
        
    def forward(self, pred_boxes, pred_logits, targets):
        total_loss = 0
        for i, (boxes, logits, target) in enumerate(zip(pred_boxes, pred_logits, targets)):
            if target['boxes'].shape[0] == 0:
                continue
            
            n = min(len(boxes), len(target['boxes']))
            if n == 0:
                continue
            
            box_loss = nn.L1Loss()(boxes[:n], target['boxes'][:n])
            class_loss = nn.CrossEntropyLoss()(logits[:n], target['labels'][:n])
            total_loss += box_loss + class_loss
        
        return total_loss / len(pred_boxes)

In [ ]:
def train_detr(model, train_loader, val_loader, epochs=20):
    optimizer = optim.AdamW(model.parameters(), lr=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, epochs)
    criterion = DETRLoss(num_classes=5)
    history = {'train_loss': [], 'val_loss': []}
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for imgs, targets in tqdm(train_loader, desc=f"DETR Epoch {epoch+1}/{epochs}"):
            imgs = imgs.to(device)
            for t in targets:
                t['boxes'] = t['boxes'].to(device)
                t['labels'] = t['labels'].to(device)
            
            optimizer.zero_grad()
            boxes, logits = model(imgs)
            loss = criterion(boxes, logits, targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.1)
            optimizer.step()
            train_loss += loss.item()
        
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for imgs, targets in val_loader:
                imgs = imgs.to(device)
                for t in targets:
                    t['boxes'] = t['boxes'].to(device)
                    t['labels'] = t['labels'].to(device)
                boxes, logits = model(imgs)
                loss = criterion(boxes, logits, targets)
                val_loss += loss.item()
        
        scheduler.step()
        avg_train = train_loss / len(train_loader)
        avg_val = val_loss / len(val_loader)
        history['train_loss'].append(avg_train)
        history['val_loss'].append(avg_val)
        print(f"DETR Epoch {epoch+1} | Train Loss: {avg_train:.4f} | Val Loss: {avg_val:.4f}")
    
    return history

In [ ]:
detr_model = DETR(num_classes=5, hidden_dim=256, num_queries=50).to(device)
print(f"DETR parameters: {sum(p.numel() for p in detr_model.parameters()):,}")

detr_history = train_detr(detr_model, train_loader, val_loader, epochs=20)

torch.save(detr_model.state_dict(), 'detr_model.pth')

In [ ]:
best_ssd = min(ssd_history['val_loss'])
best_ssd_epoch = ssd_history['val_loss'].index(best_ssd) + 1
best_detr = min(detr_history['val_loss'])
best_detr_epoch = detr_history['val_loss'].index(best_detr) + 1

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0,0].plot(ssd_history['train_loss'], label='SSD Train', linewidth=2, color='blue')
axes[0,0].plot(ssd_history['val_loss'], label='SSD Val', linewidth=2, color='blue', linestyle='--')
axes[0,0].plot(detr_history['train_loss'], label='DETR Train', linewidth=2, color='red')
axes[0,0].plot(detr_history['val_loss'], label='DETR Val', linewidth=2, color='red', linestyle='--')
axes[0,0].set_xlabel('Epoch')
axes[0,0].set_ylabel('Loss')
axes[0,0].set_title('SSD vs DETR - Loss Curves')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

metrics = ['Best Loss', 'Final Loss']
ssd_values = [best_ssd, ssd_history['val_loss'][-1]]
detr_values = [best_detr, detr_history['val_loss'][-1]]

x = np.arange(len(metrics))
width = 0.35
axes[0,1].bar(x - width/2, ssd_values, width, label='SSD', color='blue')
axes[0,1].bar(x + width/2, detr_values, width, label='DETR', color='red')
axes[0,1].set_xlabel('Metrics')
axes[0,1].set_ylabel('Loss')
axes[0,1].set_title('Model Comparison')
axes[0,1].set_xticks(x)
axes[0,1].set_xticklabels(metrics)
axes[0,1].legend()
axes[0,1].grid(True, alpha=0.3)

axes[1,0].plot(ssd_history['val_loss'], label='SSD', linewidth=2, color='blue')
axes[1,0].plot(detr_history['val_loss'], label='DETR', linewidth=2, color='red')
axes[1,0].set_xlabel('Epoch')
axes[1,0].set_ylabel('Validation Loss')
axes[1,0].set_title('Validation Loss Comparison')
axes[1,0].legend()
axes[1,0].grid(True, alpha=0.3)

axes[1,1].axis('off')
if best_detr < best_ssd:
    winner = "DETR"
    improvement = ((best_ssd - best_detr) / best_ssd) * 100
else:
    winner = "SSD"
    improvement = ((best_detr - best_ssd) / best_detr) * 100

summary_text = f"""
FINAL RESULTS

SSD:
Best Val Loss: {best_ssd:.4f} (Epoch {best_ssd_epoch})
Final Val Loss: {ssd_history['val_loss'][-1]:.4f}

DETR:
Best Val Loss: {best_detr:.4f} (Epoch {best_detr_epoch})
Final Val Loss: {detr_history['val_loss'][-1]:.4f}

WINNER: {winner}
Improvement: {improvement:.2f}%
"""

axes[1,1].text(0.1, 0.5, summary_text, fontsize=11, fontfamily='monospace', verticalalignment='center', fontweight='bold')
axes[1,1].axis('off')

plt.tight_layout()
plt.show()

print(summary_text)

In [ ]:
def calculate_map(model, val_loader, conf_thresh=0.3):
    model.eval()
    all_pred_boxes = []
    all_pred_scores = []
    all_pred_labels = []
    all_gt_boxes = []
    all_gt_labels = []
    
    with torch.no_grad():
        for imgs, targets in tqdm(val_loader, desc="Computing mAP"):
            imgs = imgs.to(device)
            locs, confs = model(imgs)
            
            scores = torch.softmax(confs, dim=-1).max(dim=-1)[0].cpu().numpy()
            labels = torch.softmax(confs, dim=-1).max(dim=-1)[1].cpu().numpy()
            boxes = locs.cpu().numpy()
            
            for i in range(len(imgs)):
                mask = scores[i] > conf_thresh
                all_pred_boxes.append(boxes[i][mask])
                all_pred_scores.append(scores[i][mask])
                all_pred_labels.append(labels[i][mask])
                
                all_gt_boxes.append(targets[i]['boxes'].numpy())
                all_gt_labels.append(targets[i]['labels'].numpy())
    
    aps = []
    for class_id in range(4):
        precisions = []
        for iou in np.arange(0.5, 1.0, 0.05):
            tp, fp, fn = 0, 0, 0
            for pred_boxes, pred_scores, pred_labels, gt_boxes, gt_labels in zip(
                all_pred_boxes, all_pred_scores, all_pred_labels, all_gt_boxes, all_gt_labels
            ):
                pred_mask = pred_labels == class_id
                gt_mask = gt_labels == class_id
                
                if pred_mask.sum() > 0 and gt_mask.sum() > 0:
                    tp += 1
                elif pred_mask.sum() > 0:
                    fp += 1
                elif gt_mask.sum() > 0:
                    fn += 1
            
            precision = tp / (tp + fp + 1e-6)
            recall = tp / (tp + fn + 1e-6)
            if recall > 0:
                precisions.append(precision)
        
        aps.append(np.mean(precisions) if precisions else 0)
    
    return np.mean(aps)

ssd_map = calculate_map(ssd_model, val_loader, conf_thresh=0.3)
detr_map = calculate_map(detr_model, val_loader, conf_thresh=0.3)

print("FINAL METRICS")
print("\n\n")
print(f"SSD mAP@0.5:0.95: {ssd_map:.4f}")
print(f"DETR mAP@0.5:0.95: {detr_map:.4f}")

if ssd_map > detr_map:
    print(f"\nSSD wins by {(ssd_map - detr_map)*100:.2f}% in mAP")
else:
    print(f"\nDETR wins by {(detr_map - ssd_map)*100:.2f}% in mAP")

In [ ]:
def visualize_predictions(model, val_loader, model_name, num_samples=4):
    model.eval()
    imgs, targets = next(iter(val_loader))
    imgs = imgs[:num_samples].to(device)
    
    with torch.no_grad():
        locs, confs = model(imgs)
        scores = torch.softmax(confs, dim=-1).max(dim=-1)[0].cpu().numpy()
        labels = torch.softmax(confs, dim=-1).max(dim=-1)[1].cpu().numpy()
        boxes = locs.cpu().numpy()
    
    class_names = ['Car', 'Van', 'Truck', 'Bus']
    fig, axes = plt.subplots(2, 2, figsize=(12, 12))
    axes = axes.flatten()
    
    for idx in range(num_samples):
        img = imgs[idx].cpu().numpy().transpose(1, 2, 0)
        img = img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
        img = np.clip(img, 0, 1)
        
        axes[idx].imshow(img)
        
        for box, score, label in zip(boxes[idx], scores[idx], labels[idx]):
            if score > 0.3 and label < 4:
                cx, cy, w, h = box
                x1 = (cx - w/2) * 640
                y1 = (cy - h/2) * 640
                rect = patches.Rectangle((x1, y1), w*640, h*640, linewidth=2, edgecolor='red', facecolor='none')
                axes[idx].add_patch(rect)
                axes[idx].text(x1, y1-5, f"{class_names[label]}: {score:.2f}", color='red', fontsize=8, fontweight='bold')
        
        axes[idx].set_title(f"{model_name} - {len([s for s in scores[idx] if s > 0.3])} objects")
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()

visualize_predictions(ssd_model, val_loader, "SSD", num_samples=4)
visualize_predictions(detr_model, val_loader, "DETR", num_samples=4)